# Comunicación con R Markdown

Este documento explica el desarrollo del proyecto de análisis exploratorio de llamadas al 911 en la CDMX, elaborado como un documento **R Markdown** ([`index.Rmd`](index.Rmd)) y publicado en la web mediante **Netlify**.

El proyecto retoma el análisis de la Unidad 3 e integra los elementos necesarios para una comunicación clara y profesional: narrativa, código reproducible, visualizaciones y referencias bibliográficas, todo en un único documento HTML autocontenido.

---
## 1. ¿Qué es R Markdown?

**R Markdown** es un formato de documento que combina texto en sintaxis Markdown con bloques de código R ejecutables (*chunks*). Al procesarse (*knitear*), genera un documento de salida —HTML, PDF o Word— que incluye el texto narrativo, el código fuente, sus resultados y las visualizaciones producidas.

Sus ventajas principales son:

| Ventaja | Descripción |
|---------|-------------|
| **Reproducibilidad** | El documento re-ejecuta todo el análisis al renderizarse, garantizando resultados actualizados |
| **Comunicación** | Integra explicaciones, código y gráficas en un solo archivo legible por cualquier audiencia |
| **Portabilidad** | El output HTML es autocontenido (`self_contained: true`): imágenes y estilos embebidos en un solo archivo |
| **Publicación web** | El HTML resultante puede alojarse directamente en servicios como Netlify sin backend ni servidor |

R Markdown es parte del ecosistema **tidyverse** y está mantenido por Posit (antes RStudio). La documentación oficial se encuentra en: [https://rmarkdown.rstudio.com](https://rmarkdown.rstudio.com)

---
## 2. Estructura del archivo `index.Rmd`

El documento R Markdown del proyecto se encuentra en `index.Rmd` dentro de esta misma carpeta. Está organizado en las siguientes secciones:

```
index.Rmd
│
├── YAML Header          ← Metadatos y configuración de salida
├── Setup chunk          ← Opciones globales de knitr
│
├── 1. Introducción
├── 2. Configuración del entorno
├── 3. Carga y preprocesamiento de datos
│
├── 4. Tarea 1 — Incidentes por mes y alcaldía
├── 5. Tarea 2 — Día de la semana con más incidentes
├── 6. Tarea 3 — Distribución horaria por categoría
├── 7. Tarea 4 — Tiempo de resolución
├── 8. Tarea 5 — Porcentaje de falsas alarmas
│
├── 9. Conclusiones
└── 10. Referencias
```

---
## 3. Encabezado YAML

El encabezado YAML define los metadatos del documento y las opciones de salida HTML. Se encuentra al inicio del archivo `index.Rmd`:

```yaml
---
title: "Análisis Exploratorio de Llamadas al 911 en la CDMX"
subtitle: "Uso de dplyr, lubridate y tidyr"
author: "Marco Sebastián Hernández Parada"
date: "`r format(Sys.Date(), '%d de %B de %Y')`"
output:
  html_document:
    theme: flatly          # Tema Bootstrap limpio
    highlight: tango       # Resaltado de sintaxis
    toc: true              # Tabla de contenidos
    toc_float: true        # TOC flotante lateral
    toc_depth: 3           # Hasta 3 niveles de encabezado
    number_sections: true  # Secciones numeradas
    self_contained: true   # HTML autocontenido (imágenes embebidas)
    code_folding: hide     # Código oculto por defecto (botón para mostrar)
    fig_width: 9
    fig_height: 5.5
---
```

La opción `self_contained: true` es clave para el despliegue en Netlify: genera un único archivo HTML sin dependencias externas.

---
## 4. Chunk de configuración global

Inmediatamente después del YAML se define el chunk `setup`, que aplica opciones globales a todos los chunks del documento:

```r
```{r setup, include=FALSE}
knitr::opts_chunk$set(
  echo      = TRUE,     # Mostrar el código fuente
  message   = FALSE,    # Suprimir mensajes de consola
  warning   = FALSE,    # Suprimir advertencias
  fig.align = "center"  # Centrar todas las figuras
)
```
```

`include=FALSE` hace que este chunk se ejecute pero no aparezca en el documento final.

---
## 5. Librerías utilizadas

El documento carga seis librerías del ecosistema tidyverse más `knitr` para tablas:

```r
library(readr)      # Lectura eficiente de CSV grandes
library(dplyr)      # Manipulación declarativa de datos
library(lubridate)  # Parseo y aritmética de fechas/horas
library(tidyr)      # Reorganización de datos al formato tidy
library(ggplot2)    # Visualizaciones de alta calidad
library(scales)     # Formato de ejes y etiquetas
library(knitr)      # Tablas formateadas en el HTML
```

Para instalarlas (solo la primera vez):

```r
install.packages(c("rmarkdown", "readr", "dplyr", "lubridate",
                   "tidyr", "ggplot2", "scales", "knitr"))
```

---
## 6. Carga y preprocesamiento de datos

Los tres archivos CSV se leen con `read_csv()` usando codificación **Latin1** (necesaria para los caracteres especiales del español en estos datos) y se combinan con `bind_rows()`:

```r
ruta <- "../Practica 3.1/dataset_911/"

df <- bind_rows(
  read_csv(paste0(ruta, "llamadas_911_2021_s1.csv"), locale = locale(encoding = "Latin1"), show_col_types = FALSE),
  read_csv(paste0(ruta, "llamadas_911_2021_s2.csv"), locale = locale(encoding = "Latin1"), show_col_types = FALSE),
  read_csv(paste0(ruta, "llamadas_911_2022_s1.csv"), locale = locale(encoding = "Latin1"), show_col_types = FALSE)
)
```

El preprocesamiento con `lubridate` genera las columnas derivadas necesarias para el análisis:

| Columna derivada | Función usada | Descripción |
|-----------------|---------------|-------------|
| `fecha_creacion_dt` | `ymd()` | Fecha de creación como objeto `Date` |
| `datetime_creacion` | `ymd_hms()` | Fecha y hora combinadas como `POSIXct` |
| `hora_del_dia` | `hour(hms())` | Hora entera (0–23) |
| `dia_semana_num` | `wday()` | Día de la semana como número (1=Lunes) |
| `dia_semana` | Vector de etiquetas | Nombre del día en español |

Adicionalmente se normaliza la columna `clas_con_f_alarma` para unificar las dos variantes de codificación de "FALTA CÍVICA" presentes en los datos originales.

---
## 7. Desarrollo de las tareas de análisis

### Tarea 1 — Incidentes por mes y alcaldía

Se aplica una doble agregación: primero se cuenta el número de incidentes por mes, alcaldía y categoría; luego se calcula el promedio mensual por grupo. Se visualizan las **8 alcaldías** con mayor volumen total mediante una gráfica de barras agrupadas con `geom_col(position = "dodge")`.

**Hallazgo principal:** Iztapalapa lidera con 4,427 delitos mensuales promedio.

---

### Tarea 2 — Día de la semana con más incidentes

Agregación simple con `group_by(dia_semana)` + `summarise(n())`. La barra del día máximo se resalta en rojo usando `scale_fill_manual()` con una condición lógica.

**Hallazgo principal:** Sábado — 325,184 llamadas.

---

### Tarea 3 — Distribución horaria por categoría

Se filtran las categorías DELITO, EMERGENCIA y URGENCIAS MEDICAS con `grepl()` y se grafica la distribución horaria con `geom_line()` + `geom_point()`, usando `scale_color_brewer()` para distinguir las categorías.

**Hallazgo principal:** Los delitos alcanzan su pico a medianoche; las urgencias médicas se mantienen elevadas durante toda la noche.

---

### Tarea 4 — Tiempo de resolución

Se calcula `difftime(datetime_cierre, datetime_creacion, units = "mins")` y se filtra a valores no negativos. El histograma se limita al **percentil 99** para excluir valores atípicos extremos (~754 días). Se añaden líneas verticales para la media y mediana con `geom_vline()`.

**Hallazgo principal:** Media = 166.32 min, Mediana = 38.9 min (fuerte sesgo a la derecha).

---

### Tarea 5 — Porcentaje de falsas alarmas

Se usa `grepl("FALSA ALARMA", ..., fixed = TRUE)` para contar los registros correspondientes y calcular el porcentaje respecto al total. La barra de falsas alarmas se resalta en rojo en la gráfica de distribución por clasificación.

**Hallazgo principal:** Solo 0.03% de las llamadas son falsas alarmas (504 de 1,671,036).

---
## 8. Renderizado del documento

Para generar el archivo HTML desde RStudio:

1. Abre `index.Rmd` en RStudio
2. Presiona el botón **Knit** (o `Ctrl+Shift+K`)
3. Se genera `index.html` en la misma carpeta

Desde la consola de R:

```r
rmarkdown::render("index.Rmd")
```

> **Nota:** El procesamiento de los 3 archivos CSV (~291 MB en total, 1,671,036 registros) puede tardar varios minutos dependiendo del equipo.

---
## 9. Publicación en Netlify

El documento se publicó en Netlify aprovechando que el HTML es **autocontenido** (`self_contained: true`): todas las imágenes, estilos y scripts están embebidos en un único archivo, lo que permite alojar el proyecto sin necesidad de un servidor o backend.

### Pasos para publicar

1. Generar `index.html` knitando `index.Rmd`
2. Ingresar a [https://app.netlify.com](https://app.netlify.com)
3. En **Sites**, arrastrar y soltar la carpeta que contiene `index.html` en la zona de *drag & drop*
4. Netlify asigna automáticamente una URL pública

### Enlace del proyecto publicado

> 🔗 [https://analisis-911.netlify.app](https://analisis-911.netlify.app)

---
## Referencias

### R Markdown y knitr

Allaire, J., Xie, Y., Dervieux, C., McPherson, J., Luraschi, J., Ushey, K., Atkins, A., Wickham, H., Cheng, J., Chang, W., & Iannone, R. (2023). *rmarkdown: Dynamic Documents for R*. R package version 2.25. https://CRAN.R-project.org/package=rmarkdown

Xie, Y., Allaire, J. J., & Grolemund, G. (2018). *R Markdown: The Definitive Guide*. Chapman and Hall/CRC. https://bookdown.org/yihui/rmarkdown/

Xie, Y. (2023). *knitr: A General-Purpose Package for Dynamic Report Generation in R*. R package version 1.45. https://CRAN.R-project.org/package=knitr

### Manipulación y análisis de datos

Wickham, H., François, R., Henry, L., Müller, K., & Vaughan, D. (2023). *dplyr: A Grammar of Data Manipulation*. R package version 1.1.4. https://CRAN.R-project.org/package=dplyr

Wickham, H., Vaughan, D., & Girlich, M. (2024). *tidyr: Tidy Messy Data*. R package version 1.3.1. https://CRAN.R-project.org/package=tidyr

Wickham, H. (2016). *ggplot2: Elegant Graphics for Data Analysis*. Springer-Verlag New York. https://ggplot2.tidyverse.org

### Manejo de fechas y tiempos

Spinu, V., Grolemund, G., & Wickham, H. (2023). *lubridate: Make Dealing with Dates a Little Easier*. R package version 1.9.3. https://CRAN.R-project.org/package=lubridate

Grolemund, G., & Wickham, H. (2011). Dates and Times Made Easy with lubridate. *Journal of Statistical Software*, 40(3), 1–25. https://www.jstatsoft.org/v40/i03/

### Ciencia de datos con R

Wickham, H., Çetinkaya-Rundel, M., & Grolemund, G. (2023). *R for Data Science* (2.ª ed.). O'Reilly Media. https://r4ds.had.co.nz/

### Dataset

Gobierno de la Ciudad de México. (2022). *Llamadas al número de atención a emergencias 911* [Conjunto de datos]. Portal de Datos Abiertos de la CDMX. https://datos.cdmx.gob.mx/dataset/llamadas-numero-de-atencion-a-emergencias-911

### Repositorio del proyecto

Hernández Parada, M. S. (2025). *Maestría en Inteligencia Artificial — Prácticas* [Repositorio de GitHub]. GitHub. https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas/blob/main/Cuatrimestre%203/5%20-%20Lenguajes%20de%20ciencia%20de%20datos%20avanzado/Practica%204.1/Comunicaci%C3%B3n%20con%20R%20Markdown.ipynb